# 04 Train YOLO Egg/Fish Detector tren Google Colab

Notebook nay train detector phu tro `egg_fish_detector.pt` bang Ultralytics YOLO. Detector chi dung de dem trung va phat hien ca, khong thay classifier 11 mon.

## 1. Clone hoac pull repo

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
from datetime import datetime

REPO_URL = "https://github.com/Vo-Minh-Tri1412/cnn-food-recognition.git"
BRANCH = "codex/colab-kaggle-workflow"
PROJECT_ROOT = Path("/content/cnn-food-recognition")

def run(cmd, cwd=None, env=None):
    print("+", " ".join(str(x) for x in cmd))
    subprocess.run([str(x) for x in cmd], cwd=cwd, env=env, check=True)

if not PROJECT_ROOT.exists():
    run(["git", "clone", "-b", BRANCH, REPO_URL, PROJECT_ROOT])
else:
    run(["git", "fetch", "origin"], cwd=PROJECT_ROOT)
    run(["git", "checkout", BRANCH], cwd=PROJECT_ROOT)
    run(["git", "pull", "origin", BRANCH], cwd=PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("BRANCH =", BRANCH)

## 2. Cai dependency va kiem tra GPU

Nen chon Colab GPU, khong chon TPU. TPU se hien CPU/no CUDA voi PyTorch/Ultralytics trong workflow nay.

In [ ]:
run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "pyyaml", "pandas"])
import torch
print("torch =", torch.__version__)
print("cuda available =", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu =", torch.cuda.get_device_name(0))
else:
    print("Canh bao: runtime nay khong co CUDA. Vao Runtime > Change runtime type > GPU.")

## 3. Mount Drive va lay dataset zip

Truoc khi chay cell nay, tren local hay chay `scripts/30_package_yolo_dataset.py` roi upload file `outputs/cloud/egg_fish_yolo.zip` len `MyDrive/canteen_checkout/datasets/egg_fish_yolo.zip`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/canteen_checkout")
DRIVE_DATASET = DRIVE_ROOT / "datasets" / "egg_fish_yolo.zip"
if not DRIVE_DATASET.exists():
    raise FileNotFoundError(f"Khong thay {DRIVE_DATASET}. Hay upload egg_fish_yolo.zip vao Drive truoc.")

RUNTIME_DATA = Path("/content/canteen_checkout_data")
RUNTIME_DATA.mkdir(parents=True, exist_ok=True)
LOCAL_ZIP = RUNTIME_DATA / "egg_fish_yolo.zip"
shutil.copy2(DRIVE_DATASET, LOCAL_ZIP)
print("Copied to", LOCAL_ZIP, "size MB", round(LOCAL_ZIP.stat().st_size / 1024 / 1024, 2))

## 4. Unzip vao runtime va audit nhanh

In [ ]:
import zipfile

DATA_ROOT = RUNTIME_DATA / "egg_fish"
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)
with zipfile.ZipFile(LOCAL_ZIP) as zf:
    zf.extractall(RUNTIME_DATA)

YAML_PATH = DATA_ROOT / "data.yaml"
print("DATA_ROOT =", DATA_ROOT)
print("YAML_PATH =", YAML_PATH)
for split in ["train", "valid", "test"]:
    img_count = len(list((DATA_ROOT / split / "images").glob("*")))
    label_count = len(list((DATA_ROOT / split / "labels").glob("*.txt")))
    print(split, "images=", img_count, "labels=", label_count)
print(YAML_PATH.read_text(encoding="utf-8"))

## 5. Train YOLO

Mac dinh dung `yolo11s.pt`. Neu muon nhanh hon thi doi `YOLO_MODEL = "yolo11n.pt"`; neu muon thu manh hon tren GPU 16GB thi doi `yolo11m.pt`.

In [ ]:
YOLO_MODEL = "yolo11s.pt"
EPOCHS = 100
IMGSZ = 640
BATCH = 16
PATIENCE = 20
RUN_NAME = "egg_fish_detector_" + datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = Path("/content/canteen_checkout_runs") / RUN_NAME

cmd = [
    sys.executable, "scripts/31_train_yolo_detector.py",
    "--data", YAML_PATH,
    "--model", YOLO_MODEL,
    "--epochs", str(EPOCHS),
    "--imgsz", str(IMGSZ),
    "--batch", str(BATCH),
    "--patience", str(PATIENCE),
    "--project", RUN_ROOT / "detector",
    "--name", "train",
    "--model-out", RUN_ROOT / "models" / "egg_fish_detector.pt",
]
run(cmd, cwd=PROJECT_ROOT)
print("RUN_ROOT =", RUN_ROOT)

## 6. Luu model va report ve Drive

In [ ]:
DRIVE_RUN = DRIVE_ROOT / "runs" / RUN_ROOT.name
DRIVE_MODEL_DIR = DRIVE_ROOT / "models"
DRIVE_RUN.mkdir(parents=True, exist_ok=True)
DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

shutil.copytree(RUN_ROOT, DRIVE_RUN, dirs_exist_ok=True)
BEST_MODEL = RUN_ROOT / "models" / "egg_fish_detector.pt"
if not BEST_MODEL.exists():
    raise FileNotFoundError(BEST_MODEL)
shutil.copy2(BEST_MODEL, DRIVE_MODEL_DIR / "egg_fish_detector.pt")
print("Saved run to", DRIVE_RUN)
print("Saved canonical detector to", DRIVE_MODEL_DIR / "egg_fish_detector.pt")

## 7. Xem nhanh metric YOLO

In [ ]:
import pandas as pd
from IPython.display import Image as IPImage, display

train_dir = next((RUN_ROOT / "detector").rglob("results.csv"), None)
if train_dir:
    df = pd.read_csv(train_dir)
    display(df.tail())
for name in ["results.png", "confusion_matrix.png", "PR_curve.png", "F1_curve.png"]:
    found = next(RUN_ROOT.rglob(name), None)
    if found:
        print(name, found)
        display(IPImage(filename=str(found), width=900))